# Employee Attrition Prediction & Retention Strategy

## 👥 Business Context

Employee turnover is costly. Replacing a skilled employee can cost up to 2x their annual salary. This analysis uses machine learning to predict which employees are at risk of leaving and identifies the key drivers of attrition (e.g., salary, overtime, job satisfaction) to inform retention strategies.

## 📊 Objectives

1. Analyze employee demographics and job satisfaction data
2. Perform Survival Analysis (Kaplan-Meier) to understand retention over time
3. Train a predictive model (Random Forest / XGBoost) to identify at-risk employees
4. Explain model predictions using SHAP values
5. Recommend targeted retention interventions

## 🔧 Methodology

- **Data**: Synthetic HR dataset (IBM HR Analytics style)
- **Techniques**: Survival Analysis, Random Forest, SHAP (SHapley Additive exPlanations)
- **Metrics**: Accuracy, Recall, ROC-AUC

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from lifelines import KaplanMeierFitter
import shap
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('mako')
%matplotlib inline

print('✓ Libraries loaded successfully')

## 1. Data Generation

Simulating a realistic HR dataset.

In [ ]:
def generate_hr_data(n=1500):
    np.random.seed(42)
    
    # Demographics
    age = np.random.normal(35, 8, n).astype(int)
    age = np.clip(age, 20, 60)
    
    # Job Info
    dept = np.random.choice(['Sales', 'R&D', 'HR'], n, p=[0.3, 0.6, 0.1])
    role = np.random.choice(['Junior', 'Senior', 'Manager', 'Director'], n, p=[0.4, 0.3, 0.2, 0.1])
    daily_rate = np.random.randint(400, 1500, n)
    distance = np.random.randint(1, 30, n)
    
    # Satisfaction & Performance
    job_satisfaction = np.random.choice([1, 2, 3, 4], n, p=[0.1, 0.2, 0.4, 0.3])
    env_satisfaction = np.random.choice([1, 2, 3, 4], n, p=[0.1, 0.2, 0.4, 0.3])
    overtime = np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7])
    
    # Tenure
    years_at_company = np.random.exponential(5, n).astype(int)
    years_since_promotion = (years_at_company * np.random.beta(2, 5, n)).astype(int)
    
    # Attrition Logic (Probabilistic)
    logit = -2.0 \
            - 0.1 * age \
            - 0.5 * job_satisfaction \
            + 1.5 * (1 if overtime == 'Yes' else 0) \
            + 0.05 * distance \
            + 0.2 * years_since_promotion
            
    # Add randomness
    prob = 1 / (1 + np.exp(-(logit + np.random.normal(0, 1, n))))
    attrition = np.random.binomial(1, prob)
    
    df = pd.DataFrame({
        'Age': age,
        'Department': dept,
        'Role': role,
        'DailyRate': daily_rate,
        'DistanceFromHome': distance,
        'JobSatisfaction': job_satisfaction,
        'EnvironmentSatisfaction': env_satisfaction,
        'OverTime': overtime,
        'YearsAtCompany': years_at_company,
        'YearsSinceLastPromotion': years_since_promotion,
        'Attrition': attrition
    })
    
    df['Attrition_Label'] = df['Attrition'].map({1: 'Yes', 0: 'No'})
    return df

df = generate_hr_data()
print(f"Dataset Shape: {df.shape}")
print(f"Attrition Rate: {df['Attrition'].mean():.2%}")
display(df.head())

## 2. Exploratory Data Analysis

Identifying factors correlated with attrition.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Attrition by Job Satisfaction
sns.countplot(x='JobSatisfaction', hue='Attrition_Label', data=df, ax=axes[0,0])
axes[0,0].set_title('Attrition by Job Satisfaction')

# Age Distribution
sns.kdeplot(data=df, x='Age', hue='Attrition_Label', fill=True, ax=axes[0,1])
axes[0,1].set_title('Age Distribution')

# Overtime Impact
sns.barplot(x='OverTime', y='Attrition', data=df, ax=axes[1,0])
axes[1,0].set_title('Attrition Probability by Overtime')

# Years Since Promotion
sns.boxplot(x='Attrition_Label', y='YearsSinceLastPromotion', data=df, ax=axes[1,1])
axes[1,1].set_title('Years Since Last Promotion')

plt.tight_layout()
plt.savefig('outputs/attrition_eda.png')
plt.show()

## 3. Survival Analysis

Using Kaplan-Meier to estimate retention rates over time.

In [ ]:
kmf = KaplanMeierFitter()

plt.figure(figsize=(12, 6))

# Fit for all employees
kmf.fit(durations=df['YearsAtCompany'], event_observed=df['Attrition'])
kmf.plot_survival_function(label='All Employees')

# Fit by Overtime status
kmf.fit(durations=df[df['OverTime']=='Yes']['YearsAtCompany'], event_observed=df[df['OverTime']=='Yes']['Attrition'])
kmf.plot_survival_function(label='Overtime: Yes')

kmf.fit(durations=df[df['OverTime']=='No']['YearsAtCompany'], event_observed=df[df['OverTime']=='No']['Attrition'])
kmf.plot_survival_function(label='Overtime: No')

plt.title('Employee Retention Curve (Kaplan-Meier)')
plt.xlabel('Years at Company')
plt.ylabel('Retention Probability')
plt.grid(True, alpha=0.3)
plt.savefig('outputs/survival_curve.png')
plt.show()

## 4. Predictive Modeling

Training a Random Forest Classifier.

In [ ]:
# Preprocessing
le = LabelEncoder()
df_model = df.copy().drop('Attrition_Label', axis=1)

for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop('Attrition', axis=1)
y = df_model['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]):.4f}")

## 5. Model Explainability (SHAP)

Understanding which features drive the predictions.

In [ ]:
# Create Tree Explainer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Summary Plot (Feature Importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, plot_type="bar", show=False)
plt.title('Feature Importance (SHAP)')
plt.savefig('outputs/shap_importance.png')
plt.show()

# Detailed Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, show=False)
plt.title('SHAP Value Impact on Attrition')
plt.savefig('outputs/shap_summary.png')
plt.show()

## 6. Conclusion & Recommendations

Actionable insights for HR.

In [ ]:
print("="*60)
print("RETENTION STRATEGY")
print("="*60)
print("1. Key Drivers: Overtime, Job Satisfaction, and Age are top predictors.")
print("2. High Risk Group: Employees working overtime with low satisfaction.")
print("3. Intervention: Implement 'No Meeting Fridays' to reduce burnout.")
print("4. Career Growth: Review promotion cycles for employees stuck at 3+ years since last promotion.")